# Movie-review sentiment classification: corrected data workflow

**Jianing Fan — course project**

This rerun notebook uses the same original CSV and DistilBERT architecture, with an exact-file checksum, duplicate checks, fixed dependency versions, and a duplicate-free training/validation split. The original notebook and its outputs remain [available separately](sentiment_prediction_system.ipynb).

**No fresh full training result is claimed here.** Data preparation and an offline tiny-model checkpoint smoke test were executed on October 19, 2026. Training and prediction cells below have intentionally empty outputs. The historical 91.38% score must not be attached to this corrected split.

[Run in Colab](https://colab.research.google.com/github/brucefjn/ML-Sentiment-Analysis-/blob/main/sentiment_prediction_system_clean.ipynb) · [Data provenance and limits](DATA.md)

## 1. Environment
Use a fresh Python 3.12 or 3.13 environment; a GPU is recommended for the full two-epoch run. The lightweight checks were run on Python 3.13.7/macOS, not Colab. In Colab the following cell downloads the repository supporting files; locally, open this notebook from the cloned repository root.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

if not Path("data_utils.py").exists():
    if "google.colab" not in sys.modules:
        raise RuntimeError("Open this notebook from the cloned repository root.")
    checkout = Path("ML-Sentiment-Analysis-")
    if not checkout.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/brucefjn/ML-Sentiment-Analysis-.git", str(checkout)], check=True)
    os.chdir(checkout)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

In [ ]:
import importlib.metadata
import json
import numpy as np
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)
from data_utils import download_original, prepare_splits, manifest

set_seed(42)
print({name: importlib.metadata.version(name) for name in
       ["numpy", "pandas", "scikit-learn", "torch", "transformers", "datasets", "accelerate"]})

## 2. Validate the original data and remove repeated text
The input contains 24,988 rows, of which 96 repeat another review text. All duplicates have consistent labels. Keep the first occurrence before the stratified 80/20 split. No replacement dataset is used, and the CSV is not redistributed by this repository.

This split is **validation**, because it is also used to select the checkpoint. It is not Stanford’s official test split. The audit covers exact full-text duplication, not every possible near-duplicate or movie-level dependence.

In [ ]:
DATA_PATH = download_original()
train_df, validation_df, data_report = prepare_splits(DATA_PATH)
assert len(train_df) == 19913 and len(validation_df) == 4979
assert not set(train_df.Review) & set(validation_df.Review)
print(json.dumps(data_report["split"], indent=2))

label2id = {"negative": 0, "positive": 1}
id2label = {value: key for key, value in label2id.items()}
train_ds = Dataset.from_pandas(train_df[["Review", "label"]], preserve_index=False)
validation_ds = Dataset.from_pandas(validation_df[["Review", "label"]], preserve_index=False)

## 3. Tokenization and model
The model revision is pinned for this new run; the original notebook did not record a revision. Text is truncated at 256 tokens, as in the course notebook. The pretrained language-model head is replaced with a two-class classification head, which must be trained.

In [ ]:
model_spec = manifest()["model"]
model_name = model_spec["id"]
revision = model_spec["revision_for_new_runs"]
tokenizer = AutoTokenizer.from_pretrained(model_name, revision=revision)

def tokenize(batch):
    return tokenizer(batch["Review"], truncation=True, padding="max_length", max_length=256)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["Review"])
validation_ds = validation_ds.map(tokenize, batched=True, remove_columns=["Review"])
train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
validation_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, revision=revision, num_labels=2, id2label=id2label, label2id=label2id
)

## 4. Train and evaluate
These cells start a full two-epoch fine-tuning run. Results belong to the new deduplicated validation split and must be reported separately from historical scores. Hardware and dependency differences can affect numerical reproducibility even with fixed seeds.

In [ ]:
def compute_metrics(eval_pred):
    logits, truth = eval_pred
    prediction = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(truth, prediction),
            "f1_macro": f1_score(truth, prediction, average="macro")}

args = TrainingArguments(
    output_dir="sentiment_bert_clean", eval_strategy="epoch", save_strategy="epoch",
    learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=16,
    num_train_epochs=2, weight_decay=0.01, load_best_model_at_end=True,
    metric_for_best_model="f1_macro", report_to="none", seed=42, data_seed=42,
    save_total_limit=2,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=validation_ds, data_collator=DataCollatorWithPadding(tokenizer),
                  compute_metrics=compute_metrics)
trainer.train()
metrics = trainer.evaluate()
print(metrics)

## 5. Verify the selected checkpoint and export
The original run logged LayerNorm key mismatches during checkpoint selection. This check compares every selected model parameter to a direct reload of the best checkpoint, then compares logits after export. It fails visibly if the checkpoint is incomplete or changes predictions. The offline smoke test only establishes that this path works for a tiny synthetic model in the pinned local environment; it cannot establish that the missing historical trained checkpoint was correct.

In [ ]:
def checked_load(path):
    restored, info = AutoModelForSequenceClassification.from_pretrained(path, output_loading_info=True)
    for key in ["missing_keys", "unexpected_keys", "mismatched_keys", "error_msgs"]:
        if info.get(key):
            raise RuntimeError(f"Checkpoint mismatch: {key}: {info[key]}")
    return restored

selected = trainer.model
reference = checked_load(trainer.state.best_model_checkpoint)
for name, value in selected.state_dict().items():
    if not torch.equal(value.detach().cpu(), reference.state_dict()[name].cpu()):
        raise RuntimeError(f"Best-checkpoint parameter differs: {name}")
del reference

export = Path("sentiment_bert_clean/export")
trainer.save_model(export)
tokenizer.save_pretrained(export)
restored = checked_load(export).to(selected.device).eval()
selected.eval()
example = tokenizer(["Absolutely fantastic — I loved it."], return_tensors="pt")
example = {key: value.to(selected.device) for key, value in example.items()}
with torch.no_grad():
    before, after = selected(**example).logits, restored(**example).logits
torch.testing.assert_close(before, after, rtol=1e-5, atol=1e-6)
model = restored
print("Checkpoint selection and save/reload checks passed.")
run_record = {"data": data_report, "model": model_spec,
              "validation_metrics": metrics,
              "packages": {name: importlib.metadata.version(name) for name in
                           ["torch", "transformers", "datasets", "accelerate", "numpy", "pandas", "scikit-learn"]}}
Path("sentiment_bert_clean/run_record.json").write_text(json.dumps(run_record, indent=2))

## 6. Inference and targeted error analysis
Softmax values are model probabilities, not calibrated certainty. The examples probe negation and sarcasm; they are not a held-out quantitative benchmark.

In [ ]:
def predict_with_proba(texts):
    if isinstance(texts, str):
        texts = [texts]
    inputs = tokenizer(texts, return_tensors="pt", truncation=True, padding=True, max_length=256)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    model.eval()
    with torch.no_grad():
        probabilities = model(**inputs).logits.softmax(dim=-1).cpu().numpy()
    for text, probs in zip(texts, probabilities):
        print({"text": text, "prediction": id2label[int(probs.argmax())],
               "probabilities": {id2label[i]: float(p) for i, p in enumerate(probs)}})

predict_with_proba(["This movie is not as good.", "Absolutely fantastic — I loved it.",
                    "I love it. Not.", "Great movie... said no one ever.",
                    "I thought it was good; it was actually terrible."])

In [ ]:
# Optional interactive demo after training and checkpoint verification.
predict_with_proba(input("Enter a movie review: "))